# Microfluidic droplet-formation solver

Interactive walkthrough of the `droplet_formation` package: closed-form Stokes field, Chebyshev verification, and equatorial force balance.

$$
\frac{1}{r}\frac{\mathrm{d}}{\mathrm{d}r}\left(r\frac{\mathrm{d}u_i}{\mathrm{d}r}\right)
=
\frac{1}{\mu_i}\frac{\partial p_i}{\partial x}
$$


In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "droplet_formation").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt

from droplet_formation import (
    Channel, mlh,
    solve_analytical, solve_chebyshev,
    forces_on_droplet, critical_diameter,
    equilibrium_diameter, real_diameter,
)

ch = Channel()          # dc = 300 μm, μ1 = 2.24 mPa·s, μ2 = 0.97 mPa·s
R, mu1, mu2 = ch.R, ch.mu1, ch.mu2
print(f"R = {R*1e6:.1f} μm,  μ1/μ2 = {mu1/mu2:.3f}")


## 1. Analytical velocity field

Closed-form profiles (10) with constants fixed by the four BCs and the two flow-rate integrals. Paper defaults for Fig. 3: $Q_1=10\,\mathrm{ml/h}$, $Q_2=1\,\mathrm{ml/h}$.

In [ ]:
Q1, Q2 = mlh(10), mlh(1)
Rd = 0.6 * R
fld = solve_analytical(R, Rd, mu1, mu2, Q1, Q2)
Q1r, Q2r = fld.reconstruct_Q()

print(f"∂p1/∂x = {fld.P1:.4e} Pa/m")
print(f"∂p2/∂x = {fld.P2:.4e} Pa/m")
print(f"U_tp   = {fld.Utp:.5f} m/s")
print(f"Q1, Q2 recovered to rel. error {(Q1r-Q1)/Q1:.2e}, {(Q2r-Q2)/Q2:.2e}")
print(f"u1(R) = {fld.u1(R):.2e}   u1(Rd) = u2(Rd) = {fld.Utp:.5f}")
print(f"shear match: μ1 u1'(Rd) = {mu1*fld.du1_dr(Rd):.4e},  μ2 u2'(Rd) = {mu2*fld.du2_dr(Rd):.4e}")
print("The two pressure gradients are different — as the paper states after eq. (19).")

### Fig. 3 — profile evolution as the droplet grows

$d_d = 0.2, 0.4, 0.6, 0.8\,d_c$. Small droplets are faster than the film on the axis; larger droplets reverse on the axis (internal vortex), matching the paper.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
for ax, frac in zip(axes, [0.2, 0.4, 0.6, 0.8]):
    f = solve_analytical(R, frac * R, mu1, mu2, mlh(10), mlh(1))
    r_in = np.linspace(0.0, f.Rd, 200)
    r_out = np.linspace(f.Rd, f.R, 200)
    ax.plot(r_in * 1e6, f.u2(r_in), color="C0", label="droplet")
    ax.plot(r_out * 1e6, f.u1(r_out), color="C1", label="film")
    ax.axvline(f.Rd * 1e6, color="k", ls="--", lw=0.8)
    ax.set_title(rf"$d_d={frac:.1f}d_c$")
    ax.set_xlabel(r"$r$ (μm)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r"$u_x$ (m/s)")
axes[0].legend(fontsize=8)
fig.suptitle(r"Fig. 3 style: $Q_1=10$ ml/h, $Q_2=1$ ml/h", y=1.03)
fig.tight_layout()
plt.show()

### Fig. 4a — effect of continuous-phase flow rate $Q_1$

Fixed $Q_2=5$ ml/h, $d_d=0.6\,d_c$. Higher $Q_1$ speeds up the film and can reverse the droplet core.

In [ ]:
Rd = 0.6 * R
plt.figure(figsize=(6.2, 4.2))
for q1, c in zip([1, 5, 10], ["C0", "C1", "C2"]):
    f = solve_analytical(R, Rd, mu1, mu2, mlh(q1), mlh(5))
    r_in = np.linspace(0.0, Rd, 250)
    r_out = np.linspace(Rd, R, 250)
    plt.plot(np.r_[r_in, r_out] * 1e6, np.r_[f.u2(r_in), f.u1(r_out)],
             color=c, label=rf"$Q_1={q1}$ ml/h")
plt.axvline(Rd * 1e6, color="r", ls="--", lw=0.9, label="interface")
plt.xlabel(r"Radial position $r$ (μm)")
plt.ylabel(r"Axial velocity $u$ (m/s)")
plt.title(r"Fig. 4a style: $Q_2=5$ ml/h, $d_d=0.6\,d_c$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Fig. 4b — effect of dispersed-phase flow rate $Q_2$

Fixed $Q_1=5$ ml/h. The film profile barely changes; the droplet core does. That is the paper’s main hydrodynamic conclusion.

In [ ]:
plt.figure(figsize=(6.2, 4.2))
for q2, c in zip([1, 5, 10], ["C0", "C1", "C2"]):
    f = solve_analytical(R, Rd, mu1, mu2, mlh(5), mlh(q2))
    r_in = np.linspace(0.0, Rd, 250)
    r_out = np.linspace(Rd, R, 250)
    plt.plot(np.r_[r_in, r_out] * 1e6, np.r_[f.u2(r_in), f.u1(r_out)],
             color=c, label=rf"$Q_2={q2}$ ml/h")
plt.axvline(Rd * 1e6, color="r", ls="--", lw=0.9, label="interface")
plt.xlabel(r"Radial position $r$ (μm)")
plt.ylabel(r"Axial velocity $u$ (m/s)")
plt.title(r"Fig. 4b style: $Q_1=5$ ml/h, $d_d=0.6\,d_c$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Chebyshev collocation of the *same* problem

Unknowns are $u_1$ on the film, $u_2$ in the droplet, and both $P_1=\partial p_1/\partial x$, $P_2=\partial p_2/\partial x$. Flow-rate integrals are enforced with Clenshaw–Curtis quadrature. This is the corrected spectral method: viscosity and pressure gradient both enter the ODE right-hand side.

In [ ]:
fld = solve_analytical(R, 0.6 * R, mu1, mu2, mlh(10), mlh(1))
chb = solve_chebyshev(R, 0.6 * R, mu1, mu2, mlh(10), mlh(1), N=50)

err1 = np.max(np.abs(chb.u1 - fld.u1(chb.r1)))
err2 = np.max(np.abs(chb.u2[1:] - fld.u2(chb.r2[1:])))
print(f"max |u1| error = {err1:.3e}")
print(f"max |u2| error = {err2:.3e}")
print(f"P1  num / ana = {chb.P1:.6e} / {fld.P1:.6e}")
print(f"P2  num / ana = {chb.P2:.6e} / {fld.P2:.6e}")
print(f"Utp num / ana = {chb.Utp:.6f} / {fld.Utp:.6f}")

plt.figure(figsize=(6.2, 4.2))
plt.plot(chb.r2 * 1e6, chb.u2, "o", ms=3, label="Chebyshev $u_2$")
plt.plot(chb.r1 * 1e6, chb.u1, "o", ms=3, label="Chebyshev $u_1$")
r = np.linspace(1e-9, R, 400)
plt.plot(r * 1e6, fld.u(r), "k-", lw=1.2, label="analytical")
plt.axvline(0.6 * R * 1e6, color="r", ls="--", lw=0.8)
plt.xlabel(r"$r$ (μm)")
plt.ylabel(r"$u$ (m/s)")
plt.title("Chebyshev vs analytical (paper BCs + $Q_1,Q_2$)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Force balance and droplet size, eqs. (30)–(43)

$$
F_\mathrm{tot}=F_{\Delta p}+F_s-F_r,\qquad
F_r=\pi d_d\sigma,\quad
F_s=\tau(R_d)\,A_d,\quad
F_{\Delta p}=(\partial p_2/\partial x)\,V_d.
$$

Equilibrium diameter $d_\mathrm{eq}$ is where $F_\mathrm{tot}=0$. Critical diameter (43) uses Cubaud’s neck-collapse time $t_\mathrm{col}=d_c^3/Q_1$:

$$
d_\mathrm{crit}=d_c\left(1-\frac{6Q_2}{\pi Q_1}\right)^{1/3}.
$$

The paper later introduces an empirical factor on $F_s$; here the equations are used as written, so the numbers are the model’s upper-bound shear, not a fitted map.

In [ ]:
ch_fig8 = Channel(sigma=2.636e-3)   # Fig. 8 / 11 value
Q1, Q2 = mlh(4.456), mlh(0.518)

dds = np.linspace(0.2 * ch_fig8.dc, 0.97 * ch_fig8.dc, 60)
Fr, Fs, Fdp, Ftot = [], [], [], []
for dd in dds:
    f = solve_analytical(ch_fig8.R, 0.5 * dd, ch_fig8.mu1, ch_fig8.mu2, Q1, Q2)
    F = forces_on_droplet(f, ch_fig8.sigma)
    Fr.append(F.Fr); Fs.append(F.Fs); Fdp.append(F.Fdp); Ftot.append(F.Ftot)

plt.figure(figsize=(6.4, 4.2))
plt.plot(dds * 1e6, np.array(Fr) * 1e6, label=r"$F_r$ (holding)")
plt.plot(dds * 1e6, np.array(Fs) * 1e6, label=r"$F_s$ (shear)")
plt.plot(dds * 1e6, np.array(Fdp) * 1e6, label=r"$F_{\Delta p}$")
plt.plot(dds * 1e6, np.array(Ftot) * 1e6, "k", label=r"$F_\mathrm{tot}$")
plt.axhline(0.0, color="k", lw=0.6)
plt.xlabel(r"Droplet diameter $d_d$ (μm)")
plt.ylabel(r"Force (μN)")
plt.title(r"Force balance vs $d_d$  ($Q_1=4.456$, $Q_2=0.518$ ml/h)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

deq = equilibrium_diameter(ch_fig8, Q1, Q2)
dcrit = critical_diameter(ch_fig8.dc, Q1, Q2)
print(f"d_eq    = {deq*1e6:.1f} μm" if deq else "d_eq not found in (0, dc)")
print(f"d_crit  = {dcrit*1e6:.1f} μm")
if deq:
    print(f"d_real  = {real_diameter(deq, ch_fig8.dc, Q1, Q2)*1e6:.1f} μm")
    print("slug flow" if deq > dcrit else "drip flow")

### Diameters versus $Q_1$ (Fig. 8 style)

In [ ]:
Q2 = mlh(0.518)
Q1s = np.linspace(1.0, 12.0, 24)
deqs, dcrits, dreals = [], [], []
for q1_mlh in Q1s:
    q1 = mlh(q1_mlh)
    deq = equilibrium_diameter(ch_fig8, q1, Q2)
    dc = critical_diameter(ch_fig8.dc, q1, Q2)
    deqs.append(None if deq is None else deq * 1e6)
    dcrits.append(dc * 1e6)
    dreals.append(None if deq is None else real_diameter(deq, ch_fig8.dc, q1, Q2) * 1e6)

plt.figure(figsize=(6.4, 4.2))
plt.plot(Q1s, deqs, "o-", ms=4, label=r"$d_\mathrm{eq}$")
plt.plot(Q1s, dcrits, "s-", ms=4, label=r"$d_\mathrm{crit}$")
plt.plot(Q1s, dreals, "^-", ms=4, label=r"$d_\mathrm{real}$")
plt.axhline(ch_fig8.dc * 1e6, color="k", ls="--", lw=0.8, label=r"$d_c$")
plt.xlabel(r"$Q_1$ (ml/h)")
plt.ylabel(r"Diameter (μm)")
plt.title(r"Fig. 8 style: $Q_2=0.518$ ml/h, $\sigma=2.636$ mN/m")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Notes versus the original notebook

| | `Project/ChebMultiPhase.ipynb` | this folder |
|---|---|---|
| ODE right-hand side | same $K$ in both phases | $K_i=(\partial p_i/\partial x)/\mu_i$ |
| How $P$ is set | prescribed $K=-1$ | from $Q_1,Q_2$ |
| $U_{tp}$ | whatever the solve gives | from shear + flow rates |
| Force balance / $d_\mathrm{crit}$ | not implemented | eqs. (30)–(43) |

A sign inconsistency in the printed eq. (21) was avoided by assembling the $6\times6$ system from the BCs and integrals instead of copying the typeset formula. Reconstructed $Q_1,Q_2$ and all four BCs close to machine precision, and the Chebyshev field matches the closed form to $\sim10^{-13}$.